# Public image products and explicit spectral analysis

This example follows the workflow attached to [AstroVIPER #374](https://github.com/casangi/astroviper/issues/374). The assumed XRADIO #661 schema uses separate polynomial coefficient and moment axes. It constructs a small coupled MFS solution so that the conversion can be checked exactly without downloading observations.

Every imager publishes flat-sky `SKY_MODEL` and coefficient-space `SKY_RESIDUAL`, including when `restore=False`. The corrected restored image is the clean-beam convolution of that model plus the reported residual. Cube finalization divides by the channel beam; MFS uses triangular polynomial division; MVC divides by its common beam and reconstructs apparent coefficients using the actual beam polynomial. A-projection remains prospective.

Continuum `PRIMARY_BEAM` has N coefficients on `frequency_polynomial`; PSF and Hessian moments have 2N−1 planes on `frequency_polynomial_moment`. The imaging drivers keep the primary beam and PSF by default; an explicit `image_data_variables_keep` list can omit them. Requesting the primary beam also retains MVC’s `COMMON_PRIMARY_BEAM`. Only the model and residual are mandatory. There is no padded frequency cube or automatic spectral diagnostic.

In [ ]:
import numpy as np
import xarray as xr
from xradio.image import make_empty_sky_image

from astroviper.processing_functions.image_analysis import spectral_products
from astroviper.processing_functions.imaging.finalize_image import (
    finalize_image_products,
    multiply_taylor_polynomials,
    public_continuum_layout,
    restore_finalized_image,
)

image = make_empty_sky_image(
    [0.0, 0.0], [32, 32], [-1e-5, 1e-5], [1e9], ["I"], [60000.0]
)
image = image.assign_coords(taylor_term=np.arange(3), psf_taylor_order=np.arange(5))
dims = ("time", "taylor_term", "polarization", "l", "m")
shape = (1, 3, 1, 32, 32)
model = xr.DataArray(np.zeros(shape), dims=dims, coords={d: image[d] for d in dims})
model.values[0, :, 0, 16, 16] = [2.0, -1.4, 1.19]
rho = xr.full_like(model, 0.001)
beam = xr.ones_like(model)
beam.values[:, 0] = 0.7
beam.values[:, 1] = -0.14
beam.values[:, 2] = 0.02
image["SKY_MODEL"] = multiply_taylor_polynomials(model, beam)
apparent_rho = multiply_taylor_polynomials(rho, beam)
x = np.linspace(-0.5, 0.5, 7)
moments = np.array([(x**order).mean() for order in range(5)])
hessian = moments[np.add.outer(np.arange(3), np.arange(3))]
image["SKY_RESIDUAL"] = apparent_rho.copy(
    data=np.einsum("ij,tjplm->tiplm", hessian, apparent_rho.values)
)
psf = np.zeros((1, 5, 1, 32, 32))
psf[0, :, 0, 16, 16] = moments
image["POINT_SPREAD_FUNCTION"] = (
    ("time", "psf_taylor_order", "polarization", "l", "m"),
    psf,
)
image["PRIMARY_BEAM_TAYLOR"] = beam
image["PRIMARY_BEAM"] = beam.isel(taylor_term=0, drop=True).expand_dims(
    frequency=image.frequency, axis=1
)
image["BEAM_FIT_PARAMS_POINT_SPREAD_FUNCTION"] = (
    ("time", "frequency", "polarization", "beam_params_label"),
    np.array([3e-5, 2e-5, 0.0]).reshape(1, 1, 1, 3),
)
image = image.assign_coords(beam_params_label=["major", "minor", "pa"])
image.attrs.update(reference_frequency=1e9, continuum_bandwidth=5e8)
# Register roles through the shared helper.
from astroviper.utils.data_group_tools import modify_data_groups_xds

modify_data_groups_xds(
    image, "model", {"sky": "SKY_MODEL"}, "Synthetic internal model."
)
modify_data_groups_xds(
    image,
    "residual",
    {
        "sky": "SKY_RESIDUAL",
        "primary_beam": "PRIMARY_BEAM",
        "point_spread_function": "POINT_SPREAD_FUNCTION",
        "beam_fit_params_point_spread_function": "BEAM_FIT_PARAMS_POINT_SPREAD_FUNCTION",
    },
    "Synthetic normal-equation right-hand sides.",
)

The imaging cycles keep their working convention. Finalization changes the public model and residual once, before optional restoration. The input remains suitable for cycle control.

In [ ]:
final = finalize_image_products(image, continuum_mode="mfs", deconvolver="mtmfs")
np.testing.assert_allclose(final.SKY_MODEL, model, atol=1e-14)
np.testing.assert_allclose(final.SKY_RESIDUAL, rho, atol=1e-14)
assert "ALPHA" not in final
restored, timing = restore_finalized_image(final, restoration_output="both")
public = public_continuum_layout(restored)
assert public.SKY_MODEL.sizes["frequency_polynomial"] == 3
assert public.POINT_SPREAD_FUNCTION.sizes["frequency_polynomial_moment"] == 5
assert public.PRIMARY_BEAM.dims == public.SKY_MODEL.dims
assert "PRIMARY_BEAM_TAYLOR" not in public
assert "ALPHA" not in public

Spectral diagnostics are a separate explicit analysis. They use flat-sky restored coefficients and the reported flat-sky residual at the same restoring beam. `ALPHA_ERROR` is a residual-based diagnostic, not a calibrated uncertainty. The diagnostic images have a singleton reference-frequency axis.

In [ ]:
analyzed = spectral_products(
    public,
    image_data_group_in_name="primary_beam_corrected_restored",
    products=("ALPHA", "ALPHA_ERROR", "BETA"),
)
assert analyzed.ALPHA.dims == ("time", "frequency", "polarization", "l", "m")
assert analyzed.sizes["frequency"] == 1
assert "ALPHA" not in public
assert np.isfinite(analyzed.ALPHA.values).any()

For distributed imaging, choose `restore` and `restoration_output` on `image_cube_single_field` or `image_continuum_single_field`; finalization happens in that same workflow. Continuum channel beams are recomputed each residual update (`wideband_primary_beam_memory_mode="recompute"`).

`weight_memory_mode` selects storage: `in_memory` retains weights in RAM, `in_place` uses a disposable output-store cache (the default), and `in_processing_set` retains completed weights in the input Processing Set. In `in_processing_set` mode, the assumed XRADIO region-I/O API owns preallocation, partition writes/reads, completion registration, provenance, metadata consolidation, and failure cleanup. Existing persisted weights are protected; every imaging run still calculates its own weights. These XRADIO APIs and the polynomial schema are development assumptions pending #661 approval; local prototypes are not part of the AstroVIPER changes.